#### Combined Analysis - Classification Windows

In [9]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import raw_data, control_data, external_data, raw_kyiv, filtered_data, daily_data

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, f1_score
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

In [11]:
base = daily_data
war_date = pd.to_datetime('2022-02-24')
pre_start = war_date - pd.DateOffset(months=6)
cv_folds = 10

In [12]:
post_windows = [
    ('0–6', war_date, war_date + pd.DateOffset(months=6)),
    ('6–12', war_date + pd.DateOffset(months=6), war_date + pd.DateOffset(months=12)),
    ('12–18', war_date + pd.DateOffset(months=12), war_date + pd.DateOffset(months=18)),
    ('18–24', war_date + pd.DateOffset(months=18), war_date + pd.DateOffset(months=24)),
    ('24–30', war_date + pd.DateOffset(months=24), war_date + pd.DateOffset(months=30))
    
]
window_order = [w[0] for w in post_windows]


In [13]:
exclude = [
    '06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f',
    '179bbc95-ec8b-4a65-98c5-5d3b566806fa',
]

#### Features

In [14]:
features = [
    # activity sent
    'donor_daily_message_count',
    'donor_daily_avg_length',
    'donor_daily_audio_count',
    'daily_active_chats',
    'donor_daily_active_hours',
    # activity received
    'n_messages_received',
    # ratio
    'words_sent_over_words_received',
    # public activity 
    'donor_daily_post_count',
    'donor_daily_comment_count',
    'donor_daily_reaction_count',
    # temporal patterns
    'night_share',
    'night_share_reactions',
    # social structure
    'frac_words_closest_5_contacts',
]

#### Models

In [15]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Linear SVM': LinearSVC(max_iter=2000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

In [16]:
dfs_to_load = [
    'donor_daily_message_count.csv',
    'daily_active_chats.csv',
    'donor_daily_audio.csv',
    'donor_daily_posts.csv',
    'donor_daily_time_shares.csv',
    'donor_daily_avg_length.csv',
    'donor_daily_comments.csv',
    'donor_daily_reactions.csv',
    'donor_daily_active_hours.csv',
    'donor_daily_messages_received.csv',
    'donor_daily_words_received.csv',
    'donor_daily_word_count.csv',
    'donor_daily_night_share_reactions.csv',
    'donor_daily_frac_words_top5.csv',
]

loaded = [pd.read_csv(base / f) for f in dfs_to_load]

for d in loaded:
    d['date'] = pd.to_datetime(d['date'])

df_raw = reduce(
    lambda l, r: pd.merge(l, r, on=['donation_id', 'date'], how='outer'),
    loaded
).fillna(0)

df_raw['words_sent_over_words_received'] = (
    df_raw['donor_daily_word_count'] / (df_raw['words_received'] + 1e-9)
)
df_raw = df_raw[~df_raw['donation_id'].isin(exclude)]

print(f'Donors: {df_raw["donation_id"].nunique()}')
print(f'Rows: {len(df_raw):,}')

Donors: 22
Rows: 37,057


In [17]:
def prepare_fold(df_window, train_dates, test_dates, features):
    train_raw = df_window[df_window['date'].isin(train_dates)].copy()
    test_raw = df_window[df_window['date'].isin(test_dates)].copy()
    train_z = train_raw.copy()
    test_z = test_raw.copy()

    for donor_id in df_window['donation_id'].unique():
        mask_train = train_raw['donation_id'] == donor_id
        mask_test  = test_raw['donation_id']  == donor_id
        if mask_train.sum() == 0:
            continue
        donor_scaler = StandardScaler()
        train_z.loc[mask_train, features] = donor_scaler.fit_transform(
            train_raw.loc[mask_train, features]
        )
        if mask_test.sum() > 0:
            test_z.loc[mask_test, features] = donor_scaler.transform(
                test_raw.loc[mask_test, features]
            )

    train_avg = train_z.groupby('date')[features + ['label']].mean().reset_index()
    test_avg  = test_z.groupby('date')[features + ['label']].mean().reset_index()

    X_train = train_avg[features].values
    y_train = (train_avg['label'] >= 0.5).astype(int).values
    X_test = test_avg[features].values
    y_test = (test_avg['label'] >= 0.5).astype(int).values
    return X_train, y_train, X_test, y_test

In [18]:
outer_cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

window_results = []

for win_label, win_start, win_end in post_windows:

    pre = df_raw[(df_raw['date'] >= pre_start) & (df_raw['date'] < war_date)].copy()
    pre['label'] = 0

    post = df_raw[(df_raw['date'] >= win_start) & (df_raw['date'] < win_end)].copy()
    post['label'] = 1

    df_window = pd.concat([pre, post]).reset_index(drop=True)

    date_labels = df_window.groupby('date')['label'].first().reset_index()
    date_labels = date_labels.sample(frac=1, random_state=42).reset_index(drop=True)
    all_dates_arr = date_labels['date'].values
    all_labels_arr = date_labels['label'].values

    for model_name, model in models.items():
        fold_accs = []
        fold_f1s = []
        y_scores_win = np.zeros(len(all_dates_arr))
        y_true_win = np.zeros(len(all_dates_arr))

        for fold_idx, (train_idx, test_idx) in enumerate(
            outer_cv.split(all_dates_arr, all_labels_arr)
        ):
            train_dates = set(all_dates_arr[train_idx])
            test_dates = set(all_dates_arr[test_idx])

            X_train, y_train, X_test, y_test = prepare_fold(
                df_window, train_dates, test_dates, features
            )
            y_true_win[test_idx] = y_test

            m = clone(model)
            m.fit(X_train, y_train)
            preds = m.predict(X_test)

            fold_accs.append((preds == y_test).mean())
            fold_f1s.append(f1_score(y_test, preds))

            if hasattr(m, 'decision_function'):
                y_scores_win[test_idx] = m.decision_function(X_test)
            else:
                y_scores_win[test_idx] = m.predict_proba(X_test)[:, 1]

        fpr, tpr, _ = roc_curve(y_true_win, y_scores_win)
        roc_auc = auc(fpr, tpr)

        window_results.append({
            'window': win_label,
            'model': model_name,
            'mean_accuracy': round(np.mean(fold_accs), 4),
            'std_accuracy': round(np.std(fold_accs), 4),
            'mean_f1': round(np.mean(fold_f1s), 4),
            'auc': round(roc_auc, 4),
        })

win_df = pd.DataFrame(window_results)
win_df['window'] = pd.Categorical(win_df['window'], categories=window_order, ordered=True)
print(f'\nTotal rows: {len(win_df)}')


Total rows: 20


#### Accuracy and F-1 Score

In [19]:
summary = win_df.groupby('window')[['mean_accuracy', 'mean_f1', 'auc']].mean().round(3)
display(summary)

,mean_accuracy,mean_f1,auc
window,,,
0–6,0.723,0.712,0.780
6–12,0.844,0.843,0.916
12–18,0.822,0.819,0.901
18–24,0.804,0.805,0.897
24–30,0.840,0.840,0.908


#### ROC curves

In [20]:
auc_pivot = win_df.pivot(index='window', columns='model', values='auc')
print('AUC per window:')
display(auc_pivot.round(3))

AUC per window:


model,Gradient Boosting,Linear SVM,Logistic Regression,Random Forest
window,,,,
0–6,0.792,0.770,0.763,0.793
6–12,0.941,0.895,0.895,0.934
12–18,0.922,0.882,0.881,0.918
18–24,0.907,0.893,0.887,0.903
24–30,0.898,0.924,0.921,0.890


#### All results

In [21]:
rows = []
for win_label in window_order:
    for model_name in models.keys():
        row = win_df[(win_df['window'] == win_label) & (win_df['model'] == model_name)].iloc[0]
        rows.append({
            'Window': win_label,
            'Model': model_name,
            'Accuracy': row['mean_accuracy'],
            'F1': row['mean_f1'],
            'AUC': row['auc'],
        })

full_df = pd.DataFrame(rows)
pivot_acc = full_df.pivot(index='Window', columns='Model', values='Accuracy')[list(models.keys())]
pivot_f1 = full_df.pivot(index='Window', columns='Model', values='F1')[list(models.keys())]
pivot_auc = full_df.pivot(index='Window', columns='Model', values='AUC')[list(models.keys())]

print('Accuracy:')
display(pivot_acc.round(3))
print('\nF1:')
display(pivot_f1.round(3))
print('\nAUC:')
display(pivot_auc.round(3))


Accuracy:


Model,Logistic Regression,Random Forest,Linear SVM,Gradient Boosting
Window,,,,
0–6,0.721,0.734,0.713,0.726
12–18,0.800,0.836,0.808,0.844
18–24,0.815,0.794,0.820,0.788
24–30,0.849,0.808,0.871,0.833
6–12,0.829,0.861,0.832,0.856



F1:


Model,Logistic Regression,Random Forest,Linear SVM,Gradient Boosting
Window,,,,
0–6,0.703,0.732,0.690,0.722
12–18,0.800,0.830,0.808,0.839
18–24,0.815,0.796,0.818,0.792
24–30,0.848,0.806,0.871,0.836
6–12,0.824,0.860,0.829,0.859



AUC:


Model,Logistic Regression,Random Forest,Linear SVM,Gradient Boosting
Window,,,,
0–6,0.763,0.793,0.770,0.792
12–18,0.881,0.918,0.882,0.922
18–24,0.887,0.903,0.893,0.907
24–30,0.921,0.890,0.924,0.898
6–12,0.895,0.934,0.895,0.941
